# AgriSense AI - PlantVillage MobileNetV2 (Fresh Colab Training)

This notebook starts a **fresh** GPU training run. It does not load or resume
`crop_disease_mobilenetv2.pt`.

Preserved training settings:
- MobileNetV2 pretrained on ImageNet
- 224x224 input
- `MobileNet_V2_Weights.DEFAULT` normalization
- Batch size 16
- AdamW learning rate 0.0005
- AdamW weight decay 0.0001
- CosineAnnealingLR
- 10 epochs
- Seed 42

Training and test cells are manual; opening or uploading the notebook does not
start training.

In [ ]:
import json
import random
import shutil
import sys
import zipfile
import hashlib
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
import uuid

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/AgriSenseAI/PlantVillageTraining")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Every run uses a unique, non-destructive folder.
RUN_ID = (
    datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    + "-"
    + uuid.uuid4().hex[:8]
)
RUN_ROOT = DRIVE_ROOT / ("run-" + RUN_ID)
RUN_ROOT.mkdir(parents=False, exist_ok=False)

# Search rather than assuming the archive location.
archive_candidates = sorted(Path("/content/drive/MyDrive").rglob("archive.zip"))
print("archive.zip candidates:")
for candidate in archive_candidates:
    print(" -", candidate)

if len(archive_candidates) == 1:
    ARCHIVE_PATH = archive_candidates[0]
elif len(archive_candidates) == 0:
    ARCHIVE_PATH = None
else:
    print("Multiple ZIP files found. Set ARCHIVE_PATH manually in the next cell.")
    ARCHIVE_PATH = None

print("Drive base:", DRIVE_ROOT)
print("Unique run folder:", RUN_ROOT)
print("Selected archive:", ARCHIVE_PATH)

In [ ]:
# Set this manually if automatic discovery found zero or multiple archive.zip files.
# Example: ARCHIVE_PATH = Path("/content/drive/MyDrive/PlantVillage/archive.zip")

if ARCHIVE_PATH is None:
    raise RuntimeError(
        "No unique archive.zip was detected. Set ARCHIVE_PATH to the exact ZIP "
        "on Google Drive, then rerun this cell."
    )

if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f"Archive not found: {ARCHIVE_PATH}")

print("Archive:", ARCHIVE_PATH)
print("Archive size GB:", round(ARCHIVE_PATH.stat().st_size / 1024**3, 3))

In [ ]:
EXTRACT_ROOT = Path("/content/PlantVillage_extracted")

with zipfile.ZipFile(ARCHIVE_PATH, "r") as archive:
    members = archive.namelist()
    bad_member = archive.testzip()
    print("ZIP member count:", len(members))
    print("First 40 members:")
    for name in members[:40]:
        print(name)
    if bad_member is not None:
        raise RuntimeError(f"Corrupt ZIP member: {bad_member}")
    print("ZIP integrity: PASSED")

    if EXTRACT_ROOT.exists() and any(EXTRACT_ROOT.iterdir()):
        print("Reusing existing extraction:", EXTRACT_ROOT)
    else:
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
        archive.extractall(EXTRACT_ROOT)
        print("Extraction completed:", EXTRACT_ROOT)

print("Extracted dataset root:", EXTRACT_ROOT)

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

def count_images(directory):
    if not directory.is_dir():
        return 0
    return sum(
        1 for path in directory.rglob("*")
        if path.is_file() and path.suffix in IMAGE_EXTENSIONS
    )

split_candidates = []
raw_candidates = []
audit_candidates = []

for candidate in EXTRACT_ROOT.rglob("*"):
    if not candidate.is_dir():
        continue

    train_dir = candidate / "train"
    test_dir = candidate / "test"
    validation_dir = candidate / "validation"
    val_dir = candidate / "val"

    if train_dir.is_dir() and test_dir.is_dir() and (
        validation_dir.is_dir() or val_dir.is_dir()
    ):
        split_candidates.append(candidate)

    direct_dirs = [path for path in candidate.iterdir() if path.is_dir()]
    image_count = count_images(candidate)
    if len(direct_dirs) >= 10 and image_count >= 1000:
        raw_candidates.append((candidate, len(direct_dirs), image_count))

for candidate in EXTRACT_ROOT.rglob("*.json"):
    if candidate.is_file():
        try:
            data = json.loads(candidate.read_text(encoding="utf-8"))
            if "class_counts" in data and "split_ratio" in data:
                audit_candidates.append((candidate, data))
        except Exception:
            pass

if DRIVE_ROOT.exists():
    for candidate in DRIVE_ROOT.rglob("*.json"):
        if candidate.is_file():
            try:
                data = json.loads(candidate.read_text(encoding="utf-8"))
                if "class_counts" in data and "split_ratio" in data:
                    audit_candidates.append((candidate, data))
            except Exception:
                pass

print("Existing split candidates:")
for candidate in sorted(set(split_candidates), key=str):
    validation = candidate / "validation"
    if not validation.is_dir():
        validation = candidate / "val"
    print(
        " -", candidate,
        "| train=", count_images(candidate / "train"),
        "| validation=", count_images(validation),
        "| test=", count_images(candidate / "test"),
    )

print("Raw class-directory candidates:")
for candidate, class_count, image_count in sorted(raw_candidates, key=lambda item: str(item[0])):
    print(" -", candidate, "| direct_dirs=", class_count, "| images=", image_count)

print("Audited split metadata candidates:")
for candidate, audit in audit_candidates:
    print(" -", candidate, "| classes=", audit.get("classes", len(audit["class_counts"])))

In [ ]:
# Explicit selection is required if discovery produces no candidate or multiple candidates.
SELECTED_SPLIT_ROOT = None
SELECTED_CLASS_ROOT = None

selected_split = None
selected_raw = None

if SELECTED_SPLIT_ROOT is not None:
    selected_split = Path(SELECTED_SPLIT_ROOT)
elif len(set(split_candidates)) == 1:
    selected_split = sorted(set(split_candidates), key=str)[0]

if SELECTED_CLASS_ROOT is not None:
    selected_raw = Path(SELECTED_CLASS_ROOT)

print("Auto/explicit split selection:", selected_split)
print("Auto/explicit raw selection:", selected_raw)

if selected_split is None and selected_raw is None:
    print("Set SELECTED_SPLIT_ROOT or SELECTED_CLASS_ROOT above from the printed candidates.")

In [ ]:
from PIL import Image

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def split_class_names(directory):
    return sorted(path.name for path in directory.iterdir() if path.is_dir())

def actual_class_counts(directory):
    counts = {}
    for class_dir in sorted(path for path in directory.iterdir() if path.is_dir()):
        counts[class_dir.name] = sum(
            1 for path in class_dir.rglob("*")
            if path.is_file() and path.suffix in IMAGE_EXTENSIONS
        )
    return counts

train_dir = None
validation_dir = None
test_dir = None
split_audit = None

if selected_split is not None:
    train_dir = selected_split / "train"
    validation_dir = selected_split / "validation"
    if not validation_dir.is_dir():
        validation_dir = selected_split / "val"
    test_dir = selected_split / "test"

    for directory in (train_dir, validation_dir, test_dir):
        if not directory.is_dir():
            raise RuntimeError(f"Missing split directory: {directory}")

    train_count = count_images(train_dir)
    validation_count = count_images(validation_dir)
    test_count = count_images(test_dir)
    if train_count == 0 or validation_count == 0 or test_count == 0:
        raise RuntimeError(
            f"Empty split detected: train={train_count}, "
            f"validation={validation_count}, test={test_count}"
        )

    train_classes = split_class_names(train_dir)
    validation_classes = split_class_names(validation_dir)
    test_classes = split_class_names(test_dir)
    if train_classes != validation_classes or train_classes != test_classes:
        raise RuntimeError(
            "Split class names differ:\n"
            f"train={train_classes}\n"
            f"validation={validation_classes}\n"
            f"test={test_classes}"
        )

    split_hashes = {}
    for split_name, split_directory in (
        ("train", train_dir),
        ("validation", validation_dir),
        ("test", test_dir),
    ):
        hashes = set()
        for image_path in split_directory.rglob("*"):
            if image_path.is_file() and image_path.suffix in IMAGE_EXTENSIONS:
                hashes.add(sha256_file(image_path))
        split_hashes[split_name] = hashes
        print(split_name, "SHA-256 unique images:", len(hashes))

    overlap_train_validation = split_hashes["train"] & split_hashes["validation"]
    overlap_train_test = split_hashes["train"] & split_hashes["test"]
    overlap_validation_test = split_hashes["validation"] & split_hashes["test"]
    if overlap_train_validation or overlap_train_test or overlap_validation_test:
        raise RuntimeError(
            "Overlapping images detected between splits: "
            f"train-validation={len(overlap_train_validation)}, "
            f"train-test={len(overlap_train_test)}, "
            f"validation-test={len(overlap_validation_test)}"
        )

    actual_train_counts = actual_class_counts(train_dir)
    actual_validation_counts = actual_class_counts(validation_dir)
    actual_test_counts = actual_class_counts(test_dir)
    actual_total_counts = {
        class_name: (
            actual_train_counts.get(class_name, 0)
            + actual_validation_counts.get(class_name, 0)
            + actual_test_counts.get(class_name, 0)
        )
        for class_name in train_classes
    }

    verified_audits = []
    for audit_path, audit in audit_candidates:
        audit_classes = set(audit.get("class_counts", {}).keys())
        audit_counts_match = (
            set(audit.get("class_counts", {}).keys()) == set(train_classes)
            and all(
                audit["class_counts"].get(class_name) == actual_total_counts.get(class_name)
                for class_name in train_classes
            )
        )
        audit_ratio = audit.get("split_ratio")
        audit_ratio_match = audit_ratio in (
            None,
            {"train": 0.7, "validation": 0.15, "test": 0.15},
        )
        if audit_classes == set(train_classes) and audit_counts_match and audit_ratio_match:
            verified_audits.append((audit_path, audit))

    if len(verified_audits) == 1:
        split_audit = verified_audits[0][1]
        print("Verified matching audit:", verified_audits[0][0])
    elif len(verified_audits) > 1:
        split_audit = verified_audits[0][1]
        print("Warning: multiple matching audits found; using the first verified audit.")
    else:
        print("Warning: no audit was verified against the actual split totals.")

    print("Verified existing split:", selected_split)
    print("Class names:", train_classes)
    print("Reused split; test set has no SHA-256 overlap.")

elif selected_raw is not None:
    from sklearn.model_selection import train_test_split

    class_dirs = sorted(path for path in selected_raw.iterdir() if path.is_dir())
    if len(class_dirs) < 10:
        raise RuntimeError("Selected raw directory does not contain at least 10 class folders.")

    rows = []
    corrupted = []
    class_counts = {}
    for class_dir in class_dirs:
        valid_paths = []
        for image_path in class_dir.rglob("*"):
            if not image_path.is_file() or image_path.suffix not in IMAGE_EXTENSIONS:
                continue
            try:
                with Image.open(image_path) as image:
                    image.verify()
                valid_paths.append(image_path)
            except Exception:
                corrupted.append(str(image_path))
        class_counts[class_dir.name] = len(valid_paths)
        rows.extend((image_path, class_dir.name) for image_path in valid_paths)

    if len(rows) < 100:
        raise RuntimeError("Too few valid images for a reproducible split.")

    paths = [row[0] for row in rows]
    labels = [row[1] for row in rows]

    train_paths, temporary_paths, train_labels, temporary_labels = train_test_split(
        paths,
        labels,
        test_size=0.30,
        random_state=SEED,
        stratify=labels,
    )
    validation_paths, test_paths, validation_labels, test_labels = train_test_split(
        temporary_paths,
        temporary_labels,
        test_size=0.50,
        random_state=SEED,
        stratify=temporary_labels,
    )

    # Use the unique run folder. Never delete processed_plantvillage.
    split_root = RUN_ROOT / "processed_plantvillage"
    split_root.mkdir(parents=False, exist_ok=False)
    train_dir = split_root / "train"
    validation_dir = split_root / "validation"
    test_dir = split_root / "test"

    for split_directory, split_paths, split_labels in (
        (train_dir, train_paths, train_labels),
        (validation_dir, validation_paths, validation_labels),
        (test_dir, test_paths, test_labels),
    ):
        for source_path, label in zip(split_paths, split_labels):
            destination = split_directory / label / source_path.name
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination)

    split_audit = {
        "source": str(selected_raw),
        "reuse_mode": "new_stratified_split",
        "seed": SEED,
        "classes": len(class_counts),
        "class_counts": class_counts,
        "corrupted_images": len(corrupted),
        "split_ratio": {"train": 0.70, "validation": 0.15, "test": 0.15},
    }
    (RUN_ROOT / "plantvillage_split_audit.json").write_text(
        json.dumps(split_audit, indent=2),
        encoding="utf-8",
    )
    print("Created reproducible stratified split:", split_root)
else:
    raise RuntimeError(
        "No split or raw class root was selected. Set SELECTED_SPLIT_ROOT or "
        "SELECTED_CLASS_ROOT in the explicit selection cell."
    )

train_count = count_images(train_dir)
validation_count = count_images(validation_dir)
test_count = count_images(test_dir)
if train_count == 0 or validation_count == 0 or test_count == 0:
    raise RuntimeError(
        f"Empty split detected: train={train_count}, "
        f"validation={validation_count}, test={test_count}"
    )

print("Train images:", train_count)
print("Validation images:", validation_count)
print("Test images:", test_count)
if split_audit is not None:
    print(json.dumps(split_audit, indent=2))

In [ ]:
subprocess_required = True
if subprocess_required:
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "tqdm", "scikit-learn", "matplotlib", "seaborn",
    ])

from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
normalization = MobileNet_V2_Weights.DEFAULT.transforms()
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.80, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(
        brightness=0.12,
        contrast=0.12,
        saturation=0.12,
    ),
    normalization,
])
evaluation_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    normalization,
])

train_data = datasets.ImageFolder(train_dir, transform=train_transform)
validation_data = datasets.ImageFolder(validation_dir, transform=evaluation_transform)
test_data = datasets.ImageFolder(test_dir, transform=evaluation_transform)

if train_data.classes != validation_data.classes or train_data.classes != test_data.classes:
    raise RuntimeError("Train, validation, and test class lists do not match.")

print("Classes:", len(train_data.classes))
print(json.dumps(train_data.classes, indent=2))
print("Split sizes:", len(train_data), len(validation_data), len(test_data))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"
batch_size = 16

train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=pin_memory,
)
validation_loader = DataLoader(
    validation_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=pin_memory,
)

# Fresh run: deliberately do not load any existing laptop checkpoint.
model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    len(train_data.classes),
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=0.0001,
)
epochs = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,
)

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Fresh model classifier classes:", model.classifier[1].out_features)

## Manual training start

Run the next cell only when ready. It is a fresh run and never loads the
existing laptop checkpoint. The best validation-loss checkpoint and history
are written to Google Drive.

In [ ]:
import os

epochs = 10
best_validation_loss = float("inf")
history = []

checkpoint_path = RUN_ROOT / "crop_disease_mobilenetv2_best.pt"
recovery_path = RUN_ROOT / "crop_disease_mobilenetv2_recovery.pt"
history_json_path = RUN_ROOT / "crop_disease_training_history.json"
history_csv_path = RUN_ROOT / "crop_disease_training_history.csv"

for protected_path in (checkpoint_path, recovery_path, history_json_path, history_csv_path):
    if protected_path.exists():
        raise RuntimeError(
            f"Protected output already exists: {protected_path}. "
            "Create a new run folder rather than overwriting it."
        )

# Default is fresh training. Set True only to resume recovery from this same run.
RESUME_FROM_RECOVERY = False
start_epoch = 0

if RESUME_FROM_RECOVERY:
    if not recovery_path.exists():
        raise RuntimeError("Recovery checkpoint was requested but no recovery file exists.")
    recovery = torch.load(recovery_path, map_location=device)
    model.load_state_dict(recovery["model_state_dict"])
    optimizer.load_state_dict(recovery["optimizer_state_dict"])
    scheduler.load_state_dict(recovery["scheduler_state_dict"])
    start_epoch = int(recovery["epoch"])
    history = recovery.get("history", [])
    best_validation_loss = float(recovery.get("best_validation_loss", "inf"))
    print("Resuming from epoch:", start_epoch)

for epoch in range(start_epoch, epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")
    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
        progress.set_postfix(loss=loss.item())

    train_loss = running_loss / len(train_data)
    train_accuracy = correct / total

    model.eval()
    validation_loss = 0.0
    validation_correct = 0
    with torch.inference_mode():
        for images, labels in validation_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            validation_loss += criterion(outputs, labels).item() * images.size(0)
            validation_correct += (outputs.argmax(dim=1) == labels).sum().item()

    validation_loss /= len(validation_data)
    validation_accuracy = validation_correct / len(validation_data)

    record = {
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "train_accuracy": float(train_accuracy),
        "validation_loss": float(validation_loss),
        "validation_accuracy": float(validation_accuracy),
        "learning_rate": float(optimizer.param_groups[0]["lr"]),
        "device": str(device),
    }
    history.append(record)
    print(json.dumps(record))

    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "epoch": epoch + 1,
        "history": history,
        "best_validation_loss": best_validation_loss,
        "class_names": train_data.classes,
        "image_size": 224,
        "normalization": "MobileNet_V2_Weights.DEFAULT",
    }, recovery_path)

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "epoch": epoch + 1,
            "best_validation_loss": best_validation_loss,
            "class_names": train_data.classes,
            "image_size": 224,
            "normalization": "MobileNet_V2_Weights.DEFAULT",
        }, checkpoint_path)

    history_json_path.write_text(json.dumps(history, indent=2), encoding="utf-8")
    pd.DataFrame(history).to_csv(history_csv_path, index=False)
    scheduler.step()

print("Recovery checkpoint:", recovery_path)
print("Best checkpoint:", checkpoint_path)

In [ ]:
# Untouched test evaluation only.
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

targets = []
predictions = []
with torch.inference_mode():
    for images, labels in tqdm(test_loader, desc="Untouched test evaluation"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        targets.extend(labels.cpu().numpy().tolist())
        predictions.extend(outputs.argmax(dim=1).cpu().numpy().tolist())

target_names = train_data.classes
cm = confusion_matrix(targets, predictions, labels=list(range(len(target_names))))
report = classification_report(
    targets,
    predictions,
    target_names=target_names,
    digits=4,
    zero_division=0,
)

metrics = {
    "dataset": "PlantVillage",
    "architecture": "MobileNetV2 transfer learning",
    "device": str(device),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seed": SEED,
    "image_size": 224,
    "batch_size": batch_size,
    "epochs_requested": epochs,
    "split_sizes": {
        "train": len(train_data),
        "validation": len(validation_data),
        "test": len(test_data),
    },
    "class_names": target_names,
    "test_accuracy": float(accuracy_score(targets, predictions)),
    "test_precision_macro": float(precision_score(targets, predictions, average="macro", zero_division=0)),
    "test_recall_macro": float(recall_score(targets, predictions, average="macro", zero_division=0)),
    "test_f1_macro": float(f1_score(targets, predictions, average="macro", zero_division=0)),
    "confusion_matrix": cm.tolist(),
}

(RUN_ROOT / "crop_disease_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(RUN_ROOT / "crop_disease_classes.json").write_text(json.dumps(target_names, indent=2), encoding="utf-8")
(RUN_ROOT / "crop_disease_classification_report.txt").write_text(report, encoding="utf-8")

print(json.dumps(metrics, indent=2))
print(report)

In [ ]:
history = json.loads(history_json_path.read_text(encoding="utf-8"))
history_frame = pd.DataFrame(history)

figure, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_frame["epoch"], history_frame["train_loss"], marker="o", label="Train loss")
axes[0].plot(history_frame["epoch"], history_frame["validation_loss"], marker="o", label="Validation loss")
axes[0].set(title="Loss by epoch", xlabel="Epoch", ylabel="Cross entropy")
axes[0].legend()

axes[1].plot(history_frame["epoch"], history_frame["train_accuracy"], marker="o", label="Train accuracy")
axes[1].plot(history_frame["epoch"], history_frame["validation_accuracy"], marker="o", label="Validation accuracy")
axes[1].set(title="Accuracy by epoch", xlabel="Epoch", ylabel="Accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig(RUN_ROOT / "crop_disease_training_curves.png", dpi=160)
plt.show()

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=target_names, yticklabels=target_names)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.title("PlantVillage Test Confusion Matrix")
plt.tight_layout()
plt.savefig(RUN_ROOT / "crop_disease_confusion_matrix.png", dpi=160)
plt.show()

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location="cpu")
export_model = mobilenet_v2(weights=None)
export_model.classifier[1] = nn.Linear(
    export_model.classifier[1].in_features,
    len(checkpoint["class_names"]),
)
export_model.load_state_dict(checkpoint["model_state_dict"])
export_model.eval()

weights_path = RUN_ROOT / "crop_disease_mobilenetv2_weights.pth"
torch.save(export_model.state_dict(), weights_path)

bundle_path = RUN_ROOT / "crop_disease_mobilenetv2.pt"
torch.save({
    "model_state_dict": export_model.state_dict(),
    "class_names": checkpoint["class_names"],
    "image_size": 224,
    "normalization": "MobileNet_V2_Weights.DEFAULT",
}, bundle_path)

print("VS Code bundle:", bundle_path)
print("Weights-only export:", weights_path)

## Download and VS Code integration

Each run is stored in a unique folder under:

`MyDrive/AgriSenseAI/PlantVillageTraining/run-YYYYMMDD-HHMMSS-UUID`

Copy these files from the run folder:
- `crop_disease_mobilenetv2.pt`
- `crop_disease_classes.json`
- `crop_disease_metrics.json`
- `crop_disease_training_history.json`
- `crop_disease_training_history.csv`
- `crop_disease_training_curves.png`
- `crop_disease_confusion_matrix.png`
- `crop_disease_classification_report.txt`
- `crop_disease_mobilenetv2_recovery.pt`

Place the backend bundle at:

`D:\agrisense\ml\saved_models\crop_disease_mobilenetv2.pt`

Keep the existing laptop checkpoint separate until the Colab test metrics and
field behavior are verified.